In [ ]:
import pyarrow as pa
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl
import timm
import torch
import numpy as np
from PIL import Image
import io
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F
from typing import List
import math
from types import SimpleNamespace


with pa.memory_map("/home/carol/chaeyeon-kim/alpha26/carla-autopilot-multimodal-dataset-test-00000-of-00049.arrow", "r") as f:
    reader = pa.ipc.open_stream(f)
    table = reader.read_all()

print(table.schema)
print(table.num_rows)
print(table[0])

In [ ]:
row = table.to_pydict()
lidar_sample = row["lidar"][0]
lidar_np = np.array(lidar_sample)
print(lidar_np.shape)   # (N, ?) 확인
print(lidar_np[0])      # 첫 번째 포인트

(4856, 4)
[-7.43413773e+01  1.87355903e-12  1.31083908e+01  7.39372909e-01]


In [ ]:
def point_cloud_to_range_view(points, H=64, W=512):
    """
    points: (N, 4) - x, y, z, intensity
    returns: (4, H, W)
    """
    x, y, z = points[:, 0], points[:, 1], points[:, 2]

    # 구면 좌표 변환
    r = np.sqrt(x**2 + y**2 + z**2)
    yaw = np.arctan2(y, x)                    # [-π, π]
    pitch = np.arcsin(z / (r + 1e-8))         # [-π/2, π/2]

    # 정규화 → pixel 좌표
    yaw_img   = (yaw   / np.pi + 1) / 2       # [0, 1]
    pitch_img = (pitch / (np.pi/2) + 1) / 2   # [0, 1]

    col = np.clip((yaw_img * (W - 1)).astype(int), 0, W - 1)
    row = np.clip(((1 - pitch_img) * (H - 1)).astype(int), 0, H - 1)

    rv = np.zeros((4, H, W), dtype=np.float32)

    rv[0, row, col] = x
    rv[1, row, col] = y
    rv[2, row, col] = z
    rv[3, row, col] = r 

    return rv

# 테스트
rv = point_cloud_to_range_view(lidar_np)
print(rv.shape)   # (4, 64, 512)

(4, 64, 512)


In [ ]:
# bytes → PIL Image
img_bytes = table.to_pydict()["image_front"][0]["bytes"]
img = Image.open(io.BytesIO(img_bytes))
print(img.size)   # (W, H)
print(img.mode)   # RGB 확인

(800, 600)
RGBA


In [5]:
img_rgb = img.convert("RGB")
print(img_rgb.mode)   # RGB
print(np.array(img_rgb).shape)   # (600, 800, 3)

RGB
(600, 800, 3)


In [6]:
from torchvision import transforms

to_tensor = transforms.ToTensor()
img_tensor = to_tensor(img_rgb)
print(img_tensor.shape)   # (3, 600, 800)

torch.Size([3, 600, 800])


In [7]:
d = table.to_pydict()

print(d["run_id"][:10])
print(d["frame"][:10])

['run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001', 'run_001']
[0, 5, 10, 15, 20, 25, 30, 35, 40, 45]


In [ ]:
#######################
# cfg
#######################
cfg = SimpleNamespace(
    RECEPTIVE_FIELD=32,
    FUTURE_HORIZON=4,

    MODEL=SimpleNamespace(
        ACTION_DIM=2,
        TRANSITION=SimpleNamespace(
            ENABLED=True,
            HIDDEN_STATE_DIM=128,
            STATE_DIM=64,
            ACTION_LATENT_DIM=32,
            USE_DROPOUT=False,
            DROPOUT_PROBABILITY=0.0,
        ),
    ),

    LOSSES=SimpleNamespace(
        WEIGHT_PROBABILISTIC=1.0,
        WEIGHT_LIDAR_RE=1.0,
        WEIGHT_VOXEL=1.0,
    ),

    OPTIMIZER=SimpleNamespace(
        LR=1e-4,
        WEIGHT_DECAY=0.01,
        ACCUMULATE_GRAD_BATCHES=1,
    ),

    SCHEDULER=SimpleNamespace(
        NAME='none',          # 'OneCycleLR' or 'none'
        PCT_START=0.3,
    ),

    STEPS=100000,
    PRETRAINED=SimpleNamespace(
        PATH=None,
    ),
)

In [ ]:
#######################
# point cloud to range view
#######################
to_tensor = transforms.ToTensor()

def point_cloud_to_range_view(points, H=64, W=512):
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    r = np.sqrt(x**2 + y**2 + z**2)
    yaw   = np.arctan2(y, x)
    pitch = np.arcsin(z / (r + 1e-8))
    yaw_img   = (yaw   / np.pi + 1) / 2
    pitch_img = (pitch / (np.pi/2) + 1) / 2
    col = np.clip((yaw_img   * (W - 1)).astype(int), 0, W - 1)
    row = np.clip(((1 - pitch_img) * (H - 1)).astype(int), 0, H - 1)
    rv = np.zeros((4, H, W), dtype=np.float32)
    depth_map = np.full((H, W), np.inf)
    for i in range(len(points)):
        r_i = r[i]
        row_i = row[i]
        col_i = col[i]

        if r_i < depth_map[row_i, col_i]:
            depth_map[row_i, col_i] = r_i
            rv[0, row_i, col_i] = x[i]
            rv[1, row_i, col_i] = y[i]
            rv[2, row_i, col_i] = z[i]
            rv[3, row_i, col_i] = r_i

    return rv

class MUVODataset(Dataset):
    def __init__(self, arrow_file, seq_len=4):
        self.seq_len = seq_len
        self.arrow_file = arrow_file

        with pa.memory_map(arrow_file, "r") as f:
            reader = pa.ipc.open_stream(f)
            table = reader.read_all()

        self.data = table.to_pydict()
        self.n = table.num_rows

        # 같은 run_id 안에서 연속 seq_len개 묶기
        self.index = []
        run_ids = self.data["run_id"]
        for i in range(0, self.n - seq_len + 1, seq_len):
             if all(run_ids[i+j] == run_ids[i] for j in range(seq_len)):
                self.index.append(i)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        i = self.index[idx]
        images, lidars, actions, speeds = [], [], [], []

        for j in range(self.seq_len):
            k = i + j
            img = Image.open(io.BytesIO(self.data["image_front"][k]["bytes"])).convert("RGB")
            images.append(to_tensor(img))

            lidar_np = np.array(self.data["lidar"][k])
            lidars.append(torch.tensor(point_cloud_to_range_view(lidar_np)))

            actions.append([self.data["throttle"][k], self.data["steer"][k]])
            speeds.append(self.data["speed_kmh"][k])

        return {
            "image":  torch.stack(images),    # (S, 3, 600, 800)
            "lidar":  torch.stack(lidars),    # (S, 4, 64, 512)
            "action": torch.tensor(actions),  # (S, 2)
            "speed":  torch.tensor(speeds),   # (S,)
            "run_id": self.data["run_id"][i]
        }

# 테스트
arrow_file = "/home/carol/chaeyeon-kim/alpha26/carla-autopilot-multimodal-dataset-test-00000-of-00049.arrow"
ds = MUVODataset(arrow_file, seq_len=4)
print(f"dataset size: {len(ds)}")

sample = ds[0]
for k, v in sample.items():
    print(k, v.shape)

dataset size: 110
image torch.Size([4, 3, 600, 800])
lidar torch.Size([4, 4, 64, 512])
action torch.Size([4, 2])
speed torch.Size([4])


In [ ]:
loader = DataLoader(ds, batch_size=1, shuffle=False)
batch = next(iter(loader))

for k, v in batch.items():
    print(k, v.shape)
# image:  (2, 4, 3, 600, 800)
# lidar:  (2, 4, 4, 64, 512)
# action: (2, 4, 2)
# speed:  (2, 4)

image torch.Size([2, 4, 3, 600, 800])
lidar torch.Size([2, 4, 4, 64, 512])
action torch.Size([2, 4, 2])
speed torch.Size([2, 4])


In [ ]:
#######################
# encoder 정의
#######################
encoder = timm.create_model(
    'resnet18',
    pretrained=True,
    features_only=True,
    out_indices=[2, 3, 4]
)
encoder.eval()

# (B, S, 3, 600, 800) → (B*S, 3, 600, 800)
B, S = 1, 4
image = batch["image"]                        # (2, 4, 3, 600, 800)
image_flat = image.view(B*S, 3, 600, 800)    # (8, 3, 600, 800)

xs = encoder(image_flat)
for i, x in enumerate(xs):
    print(f"scale {i}: {x.shape}")

scale 0: torch.Size([8, 128, 75, 100])
scale 1: torch.Size([8, 256, 38, 50])
scale 2: torch.Size([8, 512, 19, 25])


In [ ]:
#######################
# Image Encoder
#######################
class FPNDecoder(nn.Module):
    def __init__(self, feature_info, out_channels=128):
        super().__init__()
        # 가장 깊은 feature부터 시작
        self.conv1 = nn.Sequential(
            nn.Conv2d(feature_info[2], out_channels, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(True),
        )
        self.skip_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(feature_info[i], out_channels, 3, 1, 1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(True),
            )
            for i in [1, 0]  # scale2, scale1 순서
        ])
        self.out_channels = out_channels

    def forward(self, xs: List[torch.Tensor]) -> torch.Tensor:
        x = self.conv1(xs[2])  # (B*S, out_channels, 19, 25)

        for i, conv in enumerate(self.skip_convs):
            size = xs[1 - i].shape[-2:]
            x = conv(xs[1 - i]) + F.interpolate(x, size=size, mode='bilinear', align_corners=False)

        return x  # (B*S, out_channels, 75, 100)

# 테스트
feature_info = [128, 256, 512]
fpn = FPNDecoder(feature_info, out_channels=128)
fpn.eval()

with torch.no_grad():
    Fc = fpn(xs)
    print(f"Fc: {Fc.shape}")  # (8, 128, 75, 100)

Fc: torch.Size([8, 128, 75, 100])


In [ ]:
#######################
# LiDAR Encoder
#######################
lidar_encoder = timm.create_model(
    'resnet18',
    pretrained=True,
    features_only=True,
    out_indices=[2, 3, 4],
    in_chans=4          # RGB 3채널 → LiDAR 4채널
)
lidar_encoder.eval()

lidar_fpn = FPNDecoder([128, 256, 512], out_channels=128)
lidar_fpn.eval()

# (B, S, 4, 64, 512) → (B*S, 4, 64, 512)
lidar_flat = batch["lidar"].view(B*S, 4, 64, 512)

with torch.no_grad():
    lidar_xs = lidar_encoder(lidar_flat)
    for i, x in enumerate(lidar_xs):
        print(f"lidar scale {i}: {x.shape}")
    
    FL = lidar_fpn(lidar_xs)
    print(f"FL: {FL.shape}")

lidar scale 0: torch.Size([8, 128, 8, 64])
lidar scale 1: torch.Size([8, 256, 4, 32])
lidar scale 2: torch.Size([8, 512, 2, 16])
FL: torch.Size([8, 128, 8, 64])


In [ ]:
BS, C, H, W = Fc.shape
Fc = Fc.view(B, S, C, H, W)                  # (B, S, C, 75, 100)

BS, C, H, W = FL.shape
FL = FL.view(B, S, C, H, W)                  # (B, S, C, 8, 64)

Fc = Fc.permute(1, 0, 2, 3, 4).contiguous()  # (S, B, C, 75, 100)
FL = FL.permute(1, 0, 2, 3, 4).contiguous()  # (S, B, C, 8, 64)

# camera: (75, 100) -> (15, 20)
S_cam, B_cam, C_cam, H_cam, W_cam = Fc.shape
Fc = F.adaptive_avg_pool2d(
    Fc.reshape(S_cam * B_cam, C_cam, H_cam, W_cam),
    (15, 20)
).view(S_cam, B_cam, C_cam, 15, 20)

# lidar: (8, 64) -> (8, 16)
S_lidar, B_lidar, C_lidar, H_lidar, W_lidar = FL.shape
FL = F.adaptive_avg_pool2d(
    FL.reshape(S_lidar * B_lidar, C_lidar, H_lidar, W_lidar),
    (8, 16)
).view(S_lidar, B_lidar, C_lidar, 8, 16)

print("Fc:", Fc.shape)   # (4, 2, 128, 15, 20)
print("FL:", FL.shape)   # (4, 2, 128, 8, 16)

In [ ]:
#######################
# Transformer Fusion
#######################
## num_pos_feats = C // 2
class PositionEmbeddingSine(nn.Module):
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale
    # 입력 feature의 크기(H,W)에 맞는 위치 embedding을 계산하는 함수 
    
    def forward(self, tensor):
        x = tensor
        B, C, h, w = x.shape

        not_mask = torch.ones((B, h, w), device=x.device)
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)

        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t   # (B, h, w, num_pos_feats)
        pos_y = y_embed[:, :, :, None] / dim_t

        pos_x = torch.stack(
            (pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)

        pos_y = torch.stack(
            (pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4
        ).flatten(3)

        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)  # (B, C, h, w)
        return pos


class SensorFusionTransformer(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels

        self.position_encode = PositionEmbeddingSine(
            num_pos_feats=channels // 2,
            normalize=True
        )

        # sensor type embedding: 0=camera, 1=lidar
        self.type_embedding = nn.Parameter(torch.zeros(2, channels))

        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=channels,
            nhead=8,
            dropout=0.1,
            batch_first=False   # default
        )
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=6)

    def feature_to_tokens(self, x):
        # x: (B, C, H, W)
        # return: (H*W, B, C)
        return x.flatten(2).permute(2, 0, 1)

    def add_pos_and_type(self, x, sensor_type_idx):
        
        tokens = self.feature_to_tokens(x)  # (H*W, B, C)

        pos = self.position_encode(x)       # (B, C, H, W)
        pos_tokens = self.feature_to_tokens(pos)  # (H*W, B, C)

        NHW, B, C = tokens.shape

        type_embed = self.type_embedding[sensor_type_idx].view(1, 1, C)  # (1, 1, C)
        type_embed = type_embed.expand(NHW, B, C)            # (H*W, B, C)

        tokens = tokens + pos_tokens + type_embed
        return tokens

    def forward(self, cam_feat, lidar_feat):
        # cam_feat:   (B, C, Hc, Wc)
        # lidar_feat: (B, C, Hl, Wl)

        cam_tokens = self.add_pos_and_type(cam_feat, sensor_type_idx=0)
        lidar_tokens = self.add_pos_and_type(lidar_feat, sensor_type_idx=1)

        # sequence dimension으로 concat
        fused_tokens = torch.cat([cam_tokens, lidar_tokens], dim=0)
        # shape: (Nc + Nl, B, C)

        out = self.transformer_encoder(fused_tokens)
        return out
        # Transformer encoder를 통과한 뒤의 최종 토큰들

In [ ]:
#######################
# RSSM
#######################
class RepresentationModel(nn.Module):
    def __init__(self, in_channels, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.min_std = 0.1

        self.module = nn.Sequential(
            nn.Linear(in_channels, in_channels),
            nn.LeakyReLU(True),
            nn.Linear(in_channels, 2 * self.latent_dim),
        )

    def forward(self, x):
        def sigmoid2(tensor: torch.Tensor, min_value: float) -> torch.Tensor:
            return 2 * torch.sigmoid(tensor / 2) + min_value

        mu_log_sigma = self.module(x)
        mu, log_sigma = torch.split(mu_log_sigma, self.latent_dim, dim=-1)
        sigma = sigmoid2(log_sigma, self.min_std)
        return mu, sigma


class RSSM(nn.Module):
    def __init__(self, embedding_dim, action_dim, hidden_state_dim, state_dim,
                 action_latent_dim, receptive_field, use_dropout=False, dropout_probability=0.0):
        super().__init__()
        self.embedding_dim     = embedding_dim
        self.state_dim         = state_dim
        self.action_dim        = action_dim
        self.hidden_state_dim  = hidden_state_dim
        self.action_latent_dim = action_latent_dim
        self.receptive_field   = receptive_field
        self.use_dropout       = use_dropout
        self.dropout_probability = dropout_probability
        self.active_inference  = False

        self.pre_gru_net = nn.Sequential(
            nn.Linear(state_dim + action_latent_dim, hidden_state_dim),
            nn.LeakyReLU(True),
        )
        self.recurrent_model = nn.GRUCell(input_size=hidden_state_dim, hidden_size=hidden_state_dim)
        self.posterior_action_module = nn.Sequential(
            nn.Linear(action_dim, action_latent_dim), nn.LeakyReLU(True),
        )
        self.posterior = RepresentationModel(
            in_channels=hidden_state_dim + embedding_dim + action_latent_dim,
            latent_dim=state_dim,
        )
        self.prior_action_module = nn.Sequential(
            nn.Linear(action_dim, action_latent_dim), nn.LeakyReLU(True),
        )
        self.prior = RepresentationModel(
            in_channels=hidden_state_dim + action_latent_dim,
            latent_dim=state_dim,
        )

    def forward(
        self,
        input_embedding,
        action,
        h_init=None,
        s_init=None,
        use_sample=True,
        policy=None,
        continuation=False,
    ):
        output = {"prior": [], "posterior": []}
        batch_size, sequence_length, _ = input_embedding.shape

        h_t = h_init if h_init is not None else input_embedding.new_zeros(
            (batch_size, self.hidden_state_dim)
        )
        sample_t = s_init if s_init is not None else input_embedding.new_zeros(
            (batch_size, self.state_dim)
        )

        for t in range(sequence_length):
            if continuation:
                # context 뒤에서 이어지는 future sequence이면
                # 첫 future step부터 action[:, 0] 사용
                action_t = action[:, t]
            else:
                # sequence 처음부터 시작하면 t=0에는 이전 action이 없으므로 zero
                action_t = torch.zeros_like(action[:, 0]) if t == 0 else action[:, t - 1]

            output_t = self.observe_step(
                h_t,
                sample_t,
                action_t,
                input_embedding[:, t],
                use_sample=use_sample,
                policy=policy,
            )

            use_prior = (
                self.training
                and self.use_dropout
                and torch.rand(1).item() < self.dropout_probability
                and t > 0
            )

            sample_t = output_t["prior"]["sample"] if use_prior else output_t["posterior"]["sample"]
            h_t = output_t["prior"]["hidden_state"]

            for key, value in output_t.items():
                output[key].append(value)

        return self.stack_list_of_dict_tensor(output, dim=1)

    def observe_step(self, h_t, sample_t, action_t, embedding_t, use_sample=True, policy=None):
        imagine_output = self.imagine_step(h_t, sample_t, action_t, use_sample, policy=policy)
        latent_action_t = self.posterior_action_module(action_t)
        posterior_mu_t, posterior_sigma_t = self.posterior(
            torch.cat([imagine_output["hidden_state"], embedding_t, latent_action_t], dim=-1)
        )
        sample_t = self.sample_from_distribution(posterior_mu_t, posterior_sigma_t, use_sample)
        return {
            "prior": imagine_output,
            "posterior": {
                "hidden_state": imagine_output["hidden_state"],
                "sample": sample_t,
                "mu": posterior_mu_t,
                "sigma": posterior_sigma_t,
            },
        }
    def imagine_step(self, h_t, sample_t, action_t, use_sample=True, policy=None):
        if self.active_inference:
            action_t = policy(torch.cat([h_t, sample_t], dim=-1))

        latent_action_t = self.prior_action_module(action_t)

        # action을 GRU 입력에도 포함
        gru_input = torch.cat([sample_t, latent_action_t], dim=-1)
        input_t = self.pre_gru_net(gru_input)

        h_t = self.recurrent_model(input_t, h_t)

        # prior distribution에도 action 사용
        prior_mu_t, prior_sigma_t = self.prior(
            torch.cat([h_t, latent_action_t], dim=-1)
        )

        sample_t = self.sample_from_distribution(
            prior_mu_t,
            prior_sigma_t,
            use_sample,
        )

        return {
            "hidden_state": h_t,
            "sample": sample_t,
            "mu": prior_mu_t,
            "sigma": prior_sigma_t,
    }
    @staticmethod
    def sample_from_distribution(mu, sigma, use_sample):
        return mu + sigma * torch.randn_like(mu) if use_sample else mu

    @staticmethod
    def stack_list_of_dict_tensor(output, dim=1):
        new_output = {}
        for outer_key, outer_value in output.items():
            if outer_value:
                new_output[outer_key] = {
                    inner_key: torch.stack([x[inner_key] for x in outer_value], dim=dim)
                    for inner_key in outer_value[0].keys()
                }
        return new_output



In [ ]:
#######################
# Decoder
#######################
class SensorHead(nn.Module):
    def __init__(self, in_channels, out_channels, downsample_factor, sensor_type):
        super().__init__()
        self.downsample_factor = downsample_factor
        self.sensor_type = sensor_type
        self.head = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x, B, S): 
        out = self.head(x)
        _, C, H, W = out.shape
        return {
            f'{self.sensor_type}_{self.downsample_factor}': out.view(B, S, C, H, W)
        }


class SensorDecoder(nn.Module):
    """
    RGB   목표: (3, 600, 800)  constant_size=(19, 25)
    LiDAR 목표: (4, 64,  512)  constant_size=(2,  16)
    """
    def __init__(self, latent_n_channels, out_channels, constant_size, sensor_type):
        super().__init__()

        # ── 1. latent → (B*S, 256, 1, 1) ────────────────────────
        self.linear = nn.Sequential(
            nn.Linear(latent_n_channels, 256),
            nn.Unflatten(-1, (256, 1, 1))
        )

        # ── 2. 공통 베이스 ────────────────────────────────────────
        # RGB:   1x1 → 19x25 → 38x50 → 76x100 → 152x200
        # LiDAR: 1x1 → 2x16  → 4x32  → 8x64   → 16x128
        self.base_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 256, kernel_size=constant_size),
            nn.ELU(),
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1),
            nn.ELU(),
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1),
            nn.ELU(),
            nn.ConvTranspose2d(256, 256, kernel_size=4, stride=2, padding=1),
            nn.ELU(),
        )

        # ── 3. 멀티스케일 업샘플링 ───────────────────────────────
        # RGB:   152x200 → 304x400  LiDAR: 16x128 → 32x256
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ELU(),
        )
        self.head_4 = SensorHead(128, out_channels, downsample_factor=4, sensor_type=sensor_type)

        # RGB:   304x400 → 608x800  LiDAR: 32x256 → 64x512
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ELU(),
        )
        self.head_2 = SensorHead(64, out_channels, downsample_factor=2, sensor_type=sensor_type)

        # head2랑 같아서 채널수를 키울 때 사용
        self.head_1 = SensorHead(64, out_channels, downsample_factor=1, sensor_type=sensor_type)


    
    def forward(self, hidden_state, sample):
        B, S, _ = hidden_state.shape
        x = torch.cat([hidden_state, sample], dim=-1).view(B * S, -1)

        x = self.linear(x)      # (B*S, 256, 1,   1  )
        x = self.base_conv(x)   # (B*S, 256, H0,  W0 )

        x = self.up1(x)
        out_4 = self.head_4(x, B, S)

        x = self.up2(x)
        out_2 = self.head_2(x, B, S)
        out_1 = self.head_1(x, B, S)  # up3 없이 head_1도 up2 출력 사용

        outputs = {**out_4, **out_2}

        return outputs

NameError: name 'nn' is not defined

In [ ]:
#######################
# Model
#######################
class Model(nn.Module):
    def __init__(self, cfg, embedding_n_channels=128, transformer_encoder=None):
        super().__init__()
        self.cfg = cfg
        self.embedding_n_channels = embedding_n_channels
        self.receptive_field = self.cfg.RECEPTIVE_FIELD

        self.image_encoder = timm.create_model("resnet18", pretrained=True,
                                               features_only=True, out_indices=[2, 3, 4])
        self.image_fpn = FPNDecoder([128, 256, 512], out_channels=embedding_n_channels)

        self.lidar_encoder = timm.create_model("resnet18", pretrained=True,
                                               features_only=True, out_indices=[2, 3, 4], in_chans=4)
        self.lidar_fpn = FPNDecoder([128, 256, 512], out_channels=embedding_n_channels)

        self.transformer_encoder = (transformer_encoder if transformer_encoder is not None
                                    else SensorFusionTransformer(channels=embedding_n_channels))
        
        latent_dim = (self.cfg.MODEL.TRANSITION.HIDDEN_STATE_DIM +
              self.cfg.MODEL.TRANSITION.STATE_DIM)  # 128+64=192

        self.image_decoder = SensorDecoder(
            latent_n_channels=latent_dim,
            out_channels=3,
            constant_size=(19, 25),
            sensor_type='rgb'
        )
        self.lidar_decoder = SensorDecoder(
            latent_n_channels=latent_dim,
            out_channels=4,
            constant_size=(2, 16),
            sensor_type='lidar_reconstruction'
        )

        if not self.cfg.MODEL.TRANSITION.ENABLED:
            raise ValueError("cfg.MODEL.TRANSITION.ENABLED must be True.")

        self.rssm = RSSM(
            embedding_dim=embedding_n_channels,
            action_dim=self.cfg.MODEL.ACTION_DIM,
            hidden_state_dim=self.cfg.MODEL.TRANSITION.HIDDEN_STATE_DIM,
            state_dim=self.cfg.MODEL.TRANSITION.STATE_DIM,
            action_latent_dim=self.cfg.MODEL.TRANSITION.ACTION_LATENT_DIM,
            receptive_field=self.receptive_field,
            use_dropout=self.cfg.MODEL.TRANSITION.USE_DROPOUT,
            dropout_probability=self.cfg.MODEL.TRANSITION.DROPOUT_PROBABILITY,
        )

    def encode_fuse_sequence(self, image, lidar):
        B, S, C_img, H_img, W_img = image.shape
        _, _, C_lidar, H_lidar, W_lidar = lidar.shape

        image_flat = image.reshape(B * S, C_img, H_img, W_img)
        lidar_flat = lidar.reshape(B * S, C_lidar, H_lidar, W_lidar)

        cam_feat   = self.image_fpn(self.image_encoder(image_flat))
        lidar_feat = self.lidar_fpn(self.lidar_encoder(lidar_flat))

        _, C, _, _ = cam_feat.shape
        cam_feat   = F.adaptive_avg_pool2d(cam_feat,   (15, 20)).reshape(B, S, C, 15, 20)
        lidar_feat = F.adaptive_avg_pool2d(lidar_feat, (8, 16)).reshape(B, S, C, 8, 16)

        embeddings = []
        for t in range(S):
            fused_tokens = self.transformer_encoder(cam_feat[:, t], lidar_feat[:, t])
            embeddings.append(fused_tokens.mean(dim=0))  # (B, C)

        return torch.stack(embeddings, dim=1)  # (B, S, C)

    # ✅ h_init, s_init 파라미터 추가
    def forward(self, batch, h_init=None, s_init=None, deployment=False, continuation=False):
        image = batch["image"].float()
        lidar = batch["lidar"].float()

        if "action" in batch:
            action = batch["action"].float()
        elif "throttle_brake" in batch and "steering" in batch:
            action = torch.cat([batch["throttle_brake"], batch["steering"]], dim=-1).float()
        else:
            raise KeyError("batch must contain 'action' or both 'throttle_brake' and 'steering'.")

        embedding_seq = self.encode_fuse_sequence(image, lidar)

        rssm_output = self.rssm(
            input_embedding=embedding_seq,
            action=action,
            h_init=h_init,
            s_init=s_init,
            use_sample=True,
            policy=None,
            continuation=continuation,
        )

        output = {
            "prior": rssm_output["prior"],
            "posterior": rssm_output["posterior"],
        }

        # posterior decode
        ht = rssm_output["posterior"]["hidden_state"]
        st = rssm_output["posterior"]["sample"]

        if hasattr(self, "image_decoder"):
            rgb_out = self.image_decoder(ht, st)
            output.update(rgb_out)

        if hasattr(self, "lidar_decoder"):
            lidar_out = self.lidar_decoder(ht, st)
            output.update(lidar_out)

        return output, rssm_output
    def imagine(self, state_imagine, predict_action=False, future_horizon=None):
        """
        Future prediction / imagination.

        입력:
            state_imagine:
                hidden_state: (B, hidden_dim)  # 관측 구간 마지막 posterior hidden
                sample:       (B, state_dim)   # 관측 구간 마지막 posterior sample
                action 또는 throttle_brake + steering:
                    future action sequence

        출력:
            output:
                prior 기반 미래 RGB / LiDAR prediction
            rssm_output:
                prior latent sequence
        """
        h_t = state_imagine["hidden_state"]
        sample_t = state_imagine["sample"]

        device = h_t.device
        B = h_t.shape[0]

        # future action 만들기
        if "action" in state_imagine:
            action = state_imagine["action"].float().to(device)
        elif "throttle_brake" in state_imagine and "steering" in state_imagine:
            action = torch.cat(
                [
                    state_imagine["throttle_brake"].float().to(device),
                    state_imagine["steering"].float().to(device),
                ],
                dim=-1,
            )
        else:
            raise KeyError(
                "state_imagine must contain 'action' or both 'throttle_brake' and 'steering'."
            )

        if future_horizon is None:
            future_horizon = action.shape[1]

        future_horizon = min(future_horizon, action.shape[1])

        priors = {
            "hidden_state": [],
            "sample": [],
            "mu": [],
            "sigma": [],
        }

        # observation embedding 없이 prior만 rollout
        for t in range(future_horizon):
            action_t = action[:, t]

            prior_t = self.rssm.imagine_step(
                h_t=h_t,
                sample_t=sample_t,
                action_t=action_t,
                use_sample=True,
                policy=None,
            )

            h_t = prior_t["hidden_state"]
            sample_t = prior_t["sample"]

            for key in priors.keys():
                priors[key].append(prior_t[key])

        prior_seq = {
            key: torch.stack(value, dim=1)
            for key, value in priors.items()
        }

        output = {
            "prior": prior_seq
        }

        ht = prior_seq["hidden_state"]   # (B, future_horizon, hidden_dim)
        st = prior_seq["sample"]         # (B, future_horizon, state_dim)

        if hasattr(self, "image_decoder"):
            rgb_out = self.image_decoder(ht, st)
            output.update(rgb_out)

        if hasattr(self, "lidar_decoder"):
            lidar_out = self.lidar_decoder(ht, st)
            output.update(lidar_out)

        return output, {"prior": prior_seq}
    
    ## rssm_out : 다음 ht로 넘겨줄거
    ## output : loss계산할 때 사용 (prior)

In [ ]:
#######################
# Training
#######################
class WorldModelTrainer(pl.LightningModule):

    def __init__(self, cfg=None, hparams=None, lr=None, embedding_n_channels=128):

        # ✅ super().__init__() 반드시 먼저
        super().__init__()
        self.save_hyperparameters(ignore=["hparams"])

        self.cfg = cfg or hparams
        self.rf = self.cfg.RECEPTIVE_FIELD
        self.fh = self.cfg.FUTURE_HORIZON
        self._prev_run_id = None

        fusion = SensorFusionTransformer(channels=embedding_n_channels)
        self.model = Model(self.cfg, embedding_n_channels=embedding_n_channels,
                           transformer_encoder=fusion)
        self.load_pretrained_weights()

        self.lr = lr if lr is not None else getattr(self.cfg.OPTIMIZER, "LR", 1e-4)
        self.weight_probabilistic = getattr(self.cfg.LOSSES, "WEIGHT_PROBABILISTIC", 1.0)
        self.weight_lidar_re      = getattr(self.cfg.LOSSES, "WEIGHT_LIDAR_RE", 1.0)
        self.weight_rgb           = 0.1
        self.weight_voxel         = getattr(self.cfg.LOSSES, "WEIGHT_VOXEL", 1.0)

        # ✅ TBPTT hidden state — super().__init__() 이후에 초기화
        self._tbptt_h: torch.Tensor | None = None
        self._tbptt_s: torch.Tensor | None = None
    

    # ────────────────────────────────────────
    # checkpoint
    # ────────────────────────────────────────

    def load_pretrained_weights(self):
        if not hasattr(self.cfg, "PRETRAINED"):
            return
        pretrained_path = getattr(self.cfg.PRETRAINED, "PATH", None)
        if not pretrained_path or not os.path.isfile(pretrained_path):
            return
        checkpoint = torch.load(pretrained_path, map_location="cpu")
        if "state_dict" in checkpoint:
            checkpoint = checkpoint["state_dict"]
        cleaned = {(k[len("model."):] if k.startswith("model.") else k): v
                   for k, v in checkpoint.items()}
        missing, unexpected = self.model.load_state_dict(cleaned, strict=False)
        print(f"Loaded: {pretrained_path}  missing={len(missing)}  unexpected={len(unexpected)}")

    # ────────────────────────────────────────
    # batch adapter
    # ────────────────────────────────────────

    def prepare_custom_batch(self, batch):
        if "action" in batch:
            if "throttle_brake" not in batch:
                batch["throttle_brake"] = batch["action"][..., 0:1]
            if "steering" not in batch:
                batch["steering"] = batch["action"][..., 1:2]
        if "range_view_pcd_xyzd" in batch and "lidar" not in batch:
            batch["lidar"] = batch["range_view_pcd_xyzd"].float()
        if "lidar" in batch and "range_view_label_1" not in batch:
            batch["range_view_label_1"] = batch["lidar"].float()
        if "image" in batch and "rgb_label_1" not in batch:
            batch["rgb_label_1"] = batch["image"].float()
        if "image" in batch:
            if batch["image"].dtype != torch.float32:
                batch["image"] = batch["image"].float()
            if batch["image"].max() > 2.0:
                batch["image"] = batch["image"] / 255.0
        if "lidar" in batch:
            batch["lidar"] = batch["lidar"].float()
        if "action" in batch:
            batch["action"] = batch["action"].float()
        return batch

    # ────────────────────────────────────────
    # forward  (일반 추론용 — TBPTT hidden 없음)
    # ────────────────────────────────────────

    def forward(self, batch):
        batch = self.prepare_custom_batch(batch)
        # ✅ h_init, s_init 넘기지 않으면 Model.forward 안에서 zero init
        output, state_dict = self.model.forward(batch)
        return output, state_dict

    # ────────────────────────────────────────
    # losses
    # ────────────────────────────────────────

    @staticmethod
    def gaussian_kl_loss(prior, posterior, eps=1e-6):
        mu_p  = prior["mu"]
        sig_p = prior["sigma"].clamp_min(eps)
        mu_q  = posterior["mu"]
        sig_q = posterior["sigma"].clamp_min(eps)
        kl = (torch.log(sig_p / sig_q)
              + (sig_q.pow(2) + (mu_q - mu_p).pow(2)) / (2.0 * sig_p.pow(2))
              - 0.5)
        return kl.mean()

    @staticmethod
    def resize_sequence_tensor(x, target_hw, mode="bilinear"):
        b, s, c, h, w = x.shape
        x = x.reshape(b * s, c, h, w)
        x = F.interpolate(x, size=target_hw, mode=mode,
                          **({} if mode == "nearest" else {"align_corners": False}))
        return x.reshape(b, s, c, *target_hw)

    def compute_loss(self, batch, output):
        losses = {}

        # ── KL loss ──────────────────────────────────────────────
        if "prior" in output and "posterior" in output:
            losses["probabilistic"] = (self.weight_probabilistic
                                    * self.gaussian_kl_loss(output["prior"], output["posterior"]))

        # ── LiDAR reconstruction loss (멀티스케일) ───────────────
        if "range_view_label_1" in batch:
            for factor in [2, 4]:
                key = f'lidar_reconstruction_{factor}'
                if key in output:
                    pred   = output[key]
                    target = batch["range_view_label_1"]
                    discount = 1 / factor  # factor=1→1.0, factor=2→0.5, factor=4→0.25
                    if pred.shape[-2:] != target.shape[-2:]:
                        target = self.resize_sequence_tensor(target, pred.shape[-2:], mode="nearest")
                    if pred.shape[2] >= 3:
                        losses[f'lidar_xyz_{factor}'] = self.weight_lidar_re * discount * F.mse_loss(
                            pred[:, :, :3], target[:, :, :3])
                    losses[f'lidar_depth_{factor}'] = self.weight_lidar_re * discount * F.l1_loss(
                        pred[:, :, -1:], target[:, :, -1:])

        # ── RGB reconstruction loss (멀티스케일) ─────────────────
        if "rgb_label_1" in batch:
            for factor in [2, 4]:
                key = f'rgb_{factor}'
                if key in output:
                    pred   = output[key]
                    target = batch["rgb_label_1"]
                    discount = 1 / factor
                    if pred.shape[-2:] != target.shape[-2:]:
                        target = self.resize_sequence_tensor(target, pred.shape[-2:], mode="bilinear")
                    losses[key] = self.weight_rgb * discount * F.l1_loss(pred, target)

        # ── Voxel loss ───────────────────────────────────────────
        if "voxel_1" in output and "voxel_label_1" in batch:
            pred   = output["voxel_1"]
            target = batch["voxel_label_1"].long()
            if target.ndim == 6 and target.shape[2] == 1:
                target = target[:, :, 0]
            b, s, c, x, y, z = pred.shape
            losses["voxel_1"] = self.weight_voxel * F.cross_entropy(
                pred.reshape(b * s, c, x, y, z),
                target.reshape(b * s, x, y, z),
            )

        if not losses:
            raise RuntimeError("No valid loss computed.")
        return losses

    def loss_reducing(self, losses):
        return sum(losses.values())

    # ────────────────────────────────────────
    # shared step  (val / test 전용)
    # ────────────────────────────────────────

    def shared_step(self, batch, mode="val"):
        batch = self.prepare_custom_batch(batch)
        output, state_dict = self.model.forward(batch)   # zero-init h
        losses = self.compute_loss(batch, output)
        return losses, output, state_dict

    # ────────────────────────────────────────
    # training_step  — TBPTT
    # ────────────────────────────────────────

    def training_step(self, batch, batch_idx):
        batch = self.prepare_custom_batch(batch)

        B = batch["image"].shape[0]
        device = batch["image"].device
        hidden_dim = self.model.rssm.hidden_state_dim
        state_dim = self.model.rssm.state_dim

        run_id = batch["run_id"]

        # batch_size=1 기준
        current_run_id = run_id[0] if isinstance(run_id, (list, tuple)) else run_id

        is_new_run = (self._prev_run_id is None) or (current_run_id != self._prev_run_id)

        if self._tbptt_h is None or self._tbptt_s is None or  is_new_run:
            h_prev = torch.zeros(B, hidden_dim, device=device)
            s_prev = torch.zeros(B, state_dim, device=device)
        else:
            h_prev = self._tbptt_h[:B].to(device).detach()
            s_prev = self._tbptt_s[:B].to(device).detach()

        self._prev_run_id = current_run_id

        output, state_dict = self.model.forward(
            batch,
            h_init=h_prev,
            s_init=s_prev
        )

        self._tbptt_h = state_dict["posterior"]["hidden_state"][:, -1, :].detach()
        self._tbptt_s = state_dict["posterior"]["sample"][:, -1, :].detach()

        losses = self.compute_loss(batch, output)
        total_loss = self.loss_reducing(losses)

        self.log("train_loss", total_loss, prog_bar=True, on_step=True, on_epoch=True)

        return total_loss

    # ✅ 에폭 끝나면 hidden state 리셋
    def on_train_epoch_end(self):
        self._tbptt_h = None
        self._tbptt_s = None
        self._prev_run_id = None

    # ────────────────────────────────────────
    # validation / test
    # ────────────────────────────────────────

    def validation_step(self, batch, batch_idx, dataloader_idx=0):
        losses, output, state_dict = self.shared_step(batch, mode="val")
        total_loss = self.loss_reducing(losses)
        self.log("val_loss", total_loss, prog_bar=True, on_step=False, on_epoch=True)
        for name, value in losses.items():
            self.log(f"val_{name}", value, prog_bar=False, on_step=False, on_epoch=True)
        return {"val_loss": total_loss}

    def test_step(self, batch, batch_idx, dataloader_idx=0):
        losses, output, state_dict = self.shared_step(batch, mode="test")
        total_loss = self.loss_reducing(losses)
        self.log("test_loss", total_loss, prog_bar=True, on_step=False, on_epoch=True)
        for name, value in losses.items():
            self.log(f"test_{name}", value, prog_bar=False, on_step=False, on_epoch=True)
        return {"test_loss": total_loss}

    # ────────────────────────────────────────
    # optimizer
    # ────────────────────────────────────────

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=self.lr,
            weight_decay=getattr(self.cfg.OPTIMIZER, "WEIGHT_DECAY", 0.0),
        )
        scheduler_name = getattr(self.cfg.SCHEDULER, "NAME", "none")
        if scheduler_name == "OneCycleLR":
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=self.lr, total_steps=self.cfg.STEPS,
                pct_start=getattr(self.cfg.SCHEDULER, "PCT_START", 0.3),
            )
            return {"optimizer": optimizer,
                    "lr_scheduler": {"scheduler": scheduler, "interval": "step"}}
        return optimizer

In [ ]:
def main():
    # cfg는 이미 위에서 SimpleNamespace로 정의했으므로
    # get_parser, get_cfg 불필요
    
    # DataLoader 직접 사용
    arrow_file = "/home/carol/chaeyeon-kim/alpha26/carla-autopilot-multimodal-dataset-test-00000-of-00049.arrow"
    ds = MUVODataset(arrow_file, seq_len=4)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=4)

    # Model, Trainer 생성
    model = WorldModelTrainer(cfg=cfg, lr=1e-4, embedding_n_channels=128)

    # 저장 경로
    save_dir = "./logs"

    callbacks = [
        pl.callbacks.ModelSummary(),
        pl.callbacks.LearningRateMonitor(),
        pl.callbacks.ModelCheckpoint(
            dirpath=save_dir,
            every_n_train_steps=1000,
            save_top_k=-1,
        ),
    ]

    trainer = pl.Trainer(
        accelerator='auto',
        max_steps=cfg.STEPS,
        callbacks=callbacks,
        logger=pl.loggers.TensorBoardLogger(save_dir=save_dir),
        log_every_n_steps=10,
        val_check_interval=20,
        check_val_every_n_epoch=None,
        limit_val_batches=0,
        accumulate_grad_batches=cfg.OPTIMIZER.ACCUMULATE_GRAD_BATCHES,
        num_sanity_val_steps=0,
    )

    trainer.fit(model, train_dataloaders=loader, val_dataloaders=loader)


if __name__ == '__main__':
    main()